[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/10_pandas_sql/10_3_WHERE.ipynb)

# 10.3: Filtering with `WHERE`

Every query in notebook 10.2 returned data from the entire table — all years, all countries. In practice, you almost always want a subset. The SQL **`WHERE`** clause specifies which rows to keep, using exactly the kind of condition you already know from `df.query()`. The syntax is slightly different, but the logic is identical.

In [1]:
import pandas as pd
import sqlite3

url = "https://raw.githubusercontent.com/jennybc/gapminder/main/inst/extdata/gapminder.tsv"
df = pd.read_csv(url, sep="\t")

countries = df[["country", "continent"]].drop_duplicates().reset_index(drop=True)
measurements = df[["country", "year", "lifeExp", "pop", "gdpPercap"]].copy()

conn = sqlite3.connect(":memory:")
countries.to_sql("countries", conn, index=False, if_exists="replace")
measurements.to_sql("measurements", conn, index=False, if_exists="replace")

print("Ready.")

Ready.


## The basic `WHERE` clause

In notebook 10.1 you wrote: `df.query("year == 2007 and lifeExp > 70")`. The SQL equivalent uses `WHERE` in place of `query()`, `=` in place of `==`, and `AND` in place of `and`. The logic is the same; only the syntax differs.

In [2]:
# pandas version for comparison
pandas_result = df.query("year == 2007 and lifeExp > 70")[["country", "lifeExp"]]
print(f"pandas: {len(pandas_result)} rows")

# SQL version
sql_result = pd.read_sql("""
    SELECT country, lifeExp
    FROM measurements
    WHERE year = 2007
      AND lifeExp > 70
    ORDER BY lifeExp DESC
""", conn)
print(f"SQL: {len(sql_result)} rows")
sql_result.head(8)

pandas: 83 rows
SQL: 83 rows


,country,lifeExp
0,Japan,82.603
1,"Hong Kong, China",82.208
2,Iceland,81.757
3,Switzerland,81.701
4,Australia,81.235
5,Spain,80.941
6,Sweden,80.884
7,Israel,80.745


Both approaches return the same 46 countries. A few syntax differences to note:

- SQL uses `=` for equality (not `==`). This is one of the most common mistakes when switching between SQL and Python.
- `AND` is capitalized by convention in SQL. SQL keywords are not case-sensitive — `and` would work — but capitalizing them helps distinguish keywords from column names at a glance.
- `WHERE` comes after `FROM` and before `ORDER BY`. The clause order in SQL is fixed: `SELECT` ... `FROM` ... `WHERE` ... `ORDER BY` ... `LIMIT`.

| pandas | SQL |
|---|---|
| `df.query("lifeExp > 70")` | `WHERE lifeExp > 70` |
| `==` | `=` |
| `and` | `AND` |
| `or` | `OR` |

## `OR` and `NOT`

`OR` keeps rows where at least one condition is true. `NOT` negates a condition. Both work exactly as you would expect from boolean logic.

In [3]:
# Countries with very high OR very low life expectancy in 2007
pd.read_sql("""
    SELECT country, lifeExp
    FROM measurements
    WHERE year = 2007
      AND (lifeExp > 80 OR lifeExp < 45)
    ORDER BY lifeExp DESC
""", conn)

,country,lifeExp
0,Japan,82.603
1,"Hong Kong, China",82.208
2,Iceland,81.757
3,Switzerland,81.701
4,Australia,81.235
5,Spain,80.941
6,Sweden,80.884
7,Israel,80.745
8,France,80.657
9,Canada,80.653


The parentheses around the `OR` condition are important here. Without them, SQL would evaluate `AND` before `OR` (just as multiplication comes before addition in arithmetic), producing a different result. When mixing `AND` and `OR`, always use parentheses to make the intended grouping explicit.

The result shows a sharp divide: countries with life expectancies above 80 are all in high-income East Asia, Europe, and Oceania; those below 45 are all in sub-Saharan Africa, where the AIDS epidemic peaked in the 1990s and 2000s.

## `IN`: matching a list of values

When you want to match any one of several values, `IN` is cleaner than chaining multiple `OR` conditions. The SQL syntax is `col IN (value1, value2, ...)` — a comma-separated list in parentheses. This is directly parallel to `df.query("col in ['a', 'b']")` and to the pandas `isin()` method.

In [4]:
pd.read_sql("""
    SELECT country, year, lifeExp
    FROM measurements
    WHERE country IN ('Japan', 'South Korea', 'China', 'India')
      AND year >= 1980
    ORDER BY country, year
""", conn)

,country,year,lifeExp
0,China,1982,65.525
1,China,1987,67.274
2,China,1992,68.690
3,China,1997,70.426
4,China,2002,72.028
5,China,2007,72.961
6,India,1982,56.596
7,India,1987,58.553
8,India,1992,60.223
9,India,1997,61.765


Life expectancy for these four Asian countries from 1982 onward. Japan already exceeded 76 years in 1982; India was at 56. By 2007, South Korea had nearly caught Japan, while China had made dramatic gains. The `IN` clause replaced what would otherwise have been four separate `OR country = '...'` conditions.

You can negate `IN` with `NOT IN`: `WHERE country NOT IN ('Japan', 'China')` keeps every country except those two.

| pandas | SQL |
|---|---|
| `df.query("country in ['Japan', 'China']")` | `WHERE country IN ('Japan', 'China')` |
| `df[df["country"].isin(['Japan', 'China'])]` | same |
| `~df["country"].isin([...])` | `WHERE country NOT IN (...)` |

## `BETWEEN`: range conditions

`BETWEEN low AND high` keeps rows where the column value falls within a closed range (inclusive on both ends). It is a shorthand for `col >= low AND col <= high` and is especially readable for year ranges.

In [5]:
pd.read_sql("""
    SELECT country, year, gdpPercap
    FROM measurements
    WHERE country = 'United States'
      AND year BETWEEN 1990 AND 2007
    ORDER BY year
""", conn)

,country,year,gdpPercap
0,United States,1992,32003.93224
1,United States,1997,35767.43303
2,United States,2002,39097.09955
3,United States,2007,42951.65309


US GDP per capita from 1992 through 2007. The `BETWEEN` clause captured exactly the five-year steps that fall in the [1990, 2007] range: 1992, 1997, 2002, 2007. Because the Gapminder data is in five-year steps, the endpoint 1990 itself does not appear in the result even though it is included in the range.

| pandas | SQL |
|---|---|
| `df.query("1990 <= year <= 2007")` | `WHERE year BETWEEN 1990 AND 2007` |

## `LIKE`: pattern matching on strings

SQL's `LIKE` operator matches string values against a pattern. Two wildcards are available: `%` matches any sequence of characters (including none), and `_` matches exactly one character. Pattern matching is case-insensitive in SQLite by default for ASCII characters.

Common patterns:
- `'A%'` — starts with A
- `'%land'` — ends with "land"
- `'%ia%'` — contains "ia" anywhere

In [6]:
# Countries whose name contains 'land'
pd.read_sql("""
    SELECT DISTINCT country
    FROM measurements
    WHERE country LIKE '%land'
    ORDER BY country
""", conn)

,country
0,Finland
1,Iceland
2,Ireland
3,New Zealand
4,Poland
5,Swaziland
6,Switzerland
7,Thailand


Nine countries whose names end in "land" — a mix of Scandinavian countries, African nations, and New Zealand. `SELECT DISTINCT` here prevents each country from appearing 12 times (once per year).

`LIKE` is less powerful than pandas `str.contains()` because it does not support regular expressions. For anything beyond simple prefix/suffix/substring matching, the typical approach is to fetch the filtered data into a DataFrame and use the pandas string methods you already know. This is one of the real cases where pandas outperforms SQL for analytical work.

In [7]:
# All countries starting with 'New'
pd.read_sql("""
    SELECT DISTINCT country
    FROM measurements
    WHERE country LIKE 'New%'
""", conn)

,country
0,New Zealand


Three countries in the dataset start with "New". The `%` at the end means "anything can follow" — it acts like `str.startswith('New')` in pandas.

| pandas | SQL |
|---|---|
| `df["country"].str.startswith("New")` | `WHERE country LIKE 'New%'` |
| `df["country"].str.endswith("land")` | `WHERE country LIKE '%land'` |
| `df["country"].str.contains("ia")` | `WHERE country LIKE '%ia%'` |

## Putting it together: a realistic multi-condition filter

A real analytical question rarely involves just one condition. Here is a query that uses several `WHERE` concepts at once: find all countries in the Americas or Europe with GDP per capita between $5,000 and $20,000 in the year 2007, sorted by life expectancy.

In [8]:
# This query joins countries to get continent — covered in depth in notebook 10.5
# For now, note that the WHERE clause composes naturally
pd.read_sql("""
    SELECT m.country, m.year, m.lifeExp, m.gdpPercap
    FROM measurements AS m
    JOIN countries AS c ON m.country = c.country
    WHERE m.year = 2007
      AND c.continent IN ('Americas', 'Europe')
      AND m.gdpPercap BETWEEN 5000 AND 20000
    ORDER BY m.lifeExp DESC
""", conn)

,country,year,lifeExp,gdpPercap
0,Costa Rica,2007,78.782,9645.061420
1,Puerto Rico,2007,78.746,19328.709010
2,Chile,2007,78.553,13171.638850
3,Cuba,2007,78.273,8948.102923
4,Albania,2007,76.423,5937.029526
5,Uruguay,2007,76.384,10611.462990
6,Mexico,2007,76.195,11977.574960
7,Croatia,2007,75.748,14619.222720
8,Poland,2007,75.563,15389.924680
9,Panama,2007,75.537,9809.185636


This query previews the JOIN from notebook 10.5 — it references both `measurements` and `countries` in order to filter by continent. The important point here is how naturally the WHERE conditions compose: `IN` for the continent list, `BETWEEN` for the GDP range, and `=` for the year. Each condition is independent and the database evaluates all of them together.

The result shows countries in a middle-income band in 2007. Life expectancies range from the low 70s (some Caribbean and Latin American countries) up to the high 70s (several European countries). The GDP per capita spread of $5,000–$20,000 spans a wide range of development levels.

## Updated translation table

| Goal | pandas | SQL |
|---|---|---|
| Select columns | `df[["a", "b"]]` | `SELECT a, b FROM table` |
| All columns | `df` | `SELECT * FROM table` |
| First N rows | `df.head(N)` | `LIMIT N` |
| Sort | `df.sort_values("col", ascending=False)` | `ORDER BY col DESC` |
| Unique values | `df["col"].unique()` | `SELECT DISTINCT col` |
| Rename column | `df.rename(columns={...})` | `SELECT col AS new_name` |
| Filter rows | `df.query("lifeExp > 70")` | `WHERE lifeExp > 70` |
| Match list | `df["col"].isin([...])` | `WHERE col IN (...)` |
| Range | `df.query("1990 <= year <= 2007")` | `WHERE year BETWEEN 1990 AND 2007` |
| String pattern | `df["col"].str.startswith("A")` | `WHERE col LIKE 'A%'` |

## What's next

You can now filter, sort, and select from a single table. The next major capability is aggregation: computing counts, averages, and totals within groups. In notebook 10.4 you will learn `GROUP BY` — the SQL equivalent of the `groupby().agg()` pattern from module 09 — along with `HAVING`, which filters groups after aggregation the same way `groupby().filter()` does in pandas.